# Attacker Profiling via MITRE ATT&CK Campaigns

**Research Question**: How does AI-MTD decision-making change under different attacker profiles?

This notebook maps campaigns from the MITRE ATT&CK framework to parameterised attacker profiles,
then runs MTDSim simulations to compare attacker effectiveness across different MTD defender strategies.

## Design Choices & Assumptions

### Tactic-to-Phase Mapping
The 14 ATT&CK tactics are collapsed into MTDSim's 6 attack phases based on functional equivalence in the cyber kill chain:

| MTDSim Phase | ATT&CK Tactics | Rationale |
|---|---|---|
| `SCAN_HOST` | Reconnaissance, Discovery | Network-level target identification |
| `ENUM_HOST` | Resource Development, Collection | Selecting and preparing targets |
| `SCAN_PORT` | Initial Access, Execution | Probing for entry points |
| `EXPLOIT_VULN` | Privilege Escalation, Defense Evasion, Persistence | Vulnerability exploitation |
| `BRUTE_FORCE` | Credential Access | Password attacks and credential theft |
| `SCAN_NEIGHBOR` | Lateral Movement, C2, Exfiltration, Impact | Post-compromise movement |

### Parameter Derivation (from normalised capability profile C[phase] in [0,1])
- **Duration multiplier**: `1.0 - 0.5 * C[phase]` (max 50% speedup for highly capable attackers)
- **Exploit success bonus**: `0.15 * C[EXPLOIT_VULN]` (additive to base complexity probability)
- **Brute-force multiplier**: `1.0 + 2.0 * C[BRUTE_FORCE]` (up to 3x more effective)
- **Attack threshold**: `int(10 * (1.0 + 0.5 * C[SCAN_NEIGHBOR]))` (more persistent lateral movers)

### Network Configuration (constant across experiments)
- **100 nodes, 5 endpoints, 8 subnets, 4 layers** — matches the DDQN training config so the AI benchmark is valid
- Fixed network seed (42) for structural reproducibility
- Simulation time: 15000s, compromise termination at 80%

### Defender Strategies (constant benchmarks)
1. **No MTD** — pure attack baseline
2. **Random** — random MTD selection every 200s
3. **Alternative** — round-robin MTD every 200s
4. **Simultaneous** — all MTDs together every 700s
5. **MTD AI (DDQN)** — trained RL agent (if model available)

### AI Model Note
The pre-trained DDQN model was trained on **default** attacker parameters. Testing it against modified
attacker profiles tests whether the AI generalises to unseen attacker behaviours — this is a research finding, not a bug.

### Campaign Filtering
Only campaigns with >= 3 techniques are included to avoid near-default profiles.

In [ ]:
!pip install mitreattack-python pyyaml -q

In [8]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.disable(logging.CRITICAL)

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import simpy
import random
import yaml
from pathlib import Path
from tqdm.notebook import tqdm

# MTDSim core
from mtdsim.network import TimeNetwork
from mtdsim.attacker import Adversary, AttackOperation, AttackerProfile
from mtdsim.attacker.profile_generator import (
    extract_campaigns, compute_capability_profile, derive_profile_params,
    build_attacker_profile, generate_all_profiles, TACTIC_TO_PHASE, PHASES,
)
from mtdsim.defender import MTDOperation
from mtdsim.defender.techniques import (
    CompleteTopologyShuffle, IPShuffle, OSDiversity, ServiceDiversity,
)
from mtdsim.data import constants
from mtdsim.stats.evaluation import Evaluation
from mtdsim.stats.security_metric_statistics import SecurityMetricStatistics

# MITRE ATT&CK
from mitreattack.stix20 import MitreAttackData

# Try to load AI components (optional — only needed if model is available)
try:
    import tensorflow as tf
    tf.get_logger().setLevel("ERROR")
    from mtdsim.ai.mtd_ai_operation import MTDAIOperation
    AI_AVAILABLE = True
except ImportError:
    AI_AVAILABLE = False
    print("TensorFlow not available — MTD AI experiments will be skipped.")

print("All imports successful.")

ModuleNotFoundError: No module named 'mtdsim'

## 1. MITRE ATT&CK Campaign Data Extraction

Download the Enterprise ATT&CK STIX bundle and extract all campaigns with their techniques.

In [ ]:
# Download and load MITRE ATT&CK Enterprise data
# This fetches the latest STIX bundle from the MITRE CTI GitHub repository
mitre_data = MitreAttackData("enterprise-attack.json")

# Extract all campaigns with their techniques
raw_campaigns = extract_campaigns(mitre_data)
print(f"Total campaigns in ATT&CK: {len(raw_campaigns)}")

# Filter to campaigns with >= 3 techniques
MIN_TECHNIQUES = 3
campaigns = [c for c in raw_campaigns if len(c['techniques']) >= MIN_TECHNIQUES]
print(f"Campaigns with >= {MIN_TECHNIQUES} techniques: {len(campaigns)}")

# Summary table
campaign_df = pd.DataFrame([
    {
        'Campaign ID': c['campaign_id'],
        'Name': c['campaign_name'],
        'Techniques': len(c['techniques']),
        'Description': c['description'][:100] + '...' if len(c['description']) > 100 else c['description'],
    }
    for c in campaigns
]).sort_values('Techniques', ascending=False).reset_index(drop=True)

display(campaign_df)

In [ ]:
# Bar chart: campaigns by number of techniques
fig, ax = plt.subplots(figsize=(14, 6))
sorted_df = campaign_df.sort_values('Techniques', ascending=True)
ax.barh(sorted_df['Name'], sorted_df['Techniques'], color='steelblue')
ax.set_xlabel('Number of Techniques')
ax.set_title('MITRE ATT&CK Campaigns by Technique Count')
ax.axvline(x=campaign_df['Techniques'].median(), color='red', linestyle='--', alpha=0.7, label=f"Median: {campaign_df['Techniques'].median():.0f}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nStatistics: min={campaign_df['Techniques'].min()}, max={campaign_df['Techniques'].max()}, "
      f"mean={campaign_df['Techniques'].mean():.1f}, median={campaign_df['Techniques'].median():.0f}")

## 2. Capability Profile Generation

Map each campaign's techniques to MTDSim attack phases and compute normalised capability profiles.

In [ ]:
# Build attacker profiles for all campaigns
profiles = [build_attacker_profile(c) for c in campaigns]

# Create capability profile DataFrame
cap_data = []
for p in profiles:
    row = {'Campaign': p.name, 'Campaign ID': p.campaign_id, 'Techniques': len(p.techniques)}
    row.update({phase: p.capability_profile.get(phase, 0.0) for phase in PHASES})
    cap_data.append(row)

cap_df = pd.DataFrame(cap_data).set_index('Campaign')
print(f"Generated {len(profiles)} attacker profiles.")
display(cap_df.head(10))

In [ ]:
# Heatmap: campaigns x phases (capability values)
phase_cols = PHASES
heatmap_data = cap_df[phase_cols].copy()

fig, ax = plt.subplots(figsize=(12, max(8, len(heatmap_data) * 0.35)))
sns.heatmap(
    heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd',
    xticklabels=[p.replace('_', '\n') for p in phase_cols],
    yticklabels=heatmap_data.index,
    ax=ax, vmin=0, vmax=1,
)
ax.set_title('Campaign Capability Profiles (0 = no capability, 1 = strongest phase)')
plt.tight_layout()
plt.show()

In [ ]:
# Radar/spider charts for representative campaigns (top 6 by technique count)
top_profiles = sorted(profiles, key=lambda p: len(p.techniques), reverse=True)[:6]

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{'type': 'polar'}] * 3] * 2,
    subplot_titles=[p.name for p in top_profiles],
)

for i, profile in enumerate(top_profiles):
    row, col = i // 3 + 1, i % 3 + 1
    values = [profile.capability_profile.get(phase, 0) for phase in PHASES]
    values.append(values[0])  # close the polygon
    labels = [p.replace('_', ' ') for p in PHASES] + [PHASES[0].replace('_', ' ')]

    fig.add_trace(
        go.Scatterpolar(r=values, theta=labels, fill='toself', name=profile.name),
        row=row, col=col,
    )

fig.update_layout(
    title_text="Capability Profiles for Top Campaigns (by technique count)",
    height=700, showlegend=False,
)
fig.update_polars(radialaxis=dict(range=[0, 1]))
fig.show()

In [ ]:
# PCA clustering of campaign capability profiles
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = cap_df[PHASES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    'PC1': X_pca[:, 0],
    'PC2': X_pca[:, 1],
    'Campaign': cap_df.index,
    'Techniques': cap_df['Techniques'].values,
})

fig = px.scatter(
    pca_df, x='PC1', y='PC2', text='Campaign', size='Techniques',
    title=f'PCA of Campaign Capability Profiles (explained variance: {pca.explained_variance_ratio_.sum():.1%})',
    labels={'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})'},
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=600, width=900)
fig.show()

## 3. Derived Simulation Parameters

Show how capability profiles translate into concrete simulation parameters.

In [ ]:
# Derived parameter summary table
param_data = []
for p in profiles:
    row = {
        'Campaign': p.name,
        'Exploit Bonus': p.exploit_success_bonus,
        'BF Multiplier': p.brute_force_multiplier,
        'Threshold': p.attack_threshold,
    }
    # Add effective durations (base * multiplier)
    for phase in ['SCAN_HOST', 'SCAN_PORT', 'EXPLOIT_VULN', 'BRUTE_FORCE']:
        row[f'{phase} dur (s)'] = round(p.get_attack_duration(phase), 1)
    param_data.append(row)

param_df = pd.DataFrame(param_data).set_index('Campaign')
display(param_df)

# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(param_df['Exploit Bonus'], bins=15, color='coral', edgecolor='black')
axes[0, 0].set_title('Exploit Success Bonus Distribution')
axes[0, 0].set_xlabel('Bonus (additive probability)')

axes[0, 1].hist(param_df['BF Multiplier'], bins=15, color='skyblue', edgecolor='black')
axes[0, 1].set_title('Brute-Force Multiplier Distribution')
axes[0, 1].set_xlabel('Multiplier')

axes[1, 0].hist(param_df['Threshold'], bins=15, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Attack Threshold Distribution')
axes[1, 0].set_xlabel('Max attempts per host')

axes[1, 1].hist(param_df['EXPLOIT_VULN dur (s)'], bins=15, color='plum', edgecolor='black')
axes[1, 1].set_title('Effective Exploit Duration Distribution')
axes[1, 1].set_xlabel('Duration (seconds)')

plt.suptitle('Distribution of Derived Parameters Across Campaigns', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Save all profiles to YAML for persistent storage
PROFILE_DIR = Path("../src/mtdsim/data/attacker_profiles")
PROFILE_DIR.mkdir(parents=True, exist_ok=True)

for p in profiles:
    safe_name = f"{p.campaign_id}_{p.name.replace(' ', '_').replace('/', '_')}"
    p.to_yaml(str(PROFILE_DIR / f"{safe_name}.yaml"))

# Write index
index = [{'campaign_id': p.campaign_id, 'name': p.name, 'num_techniques': len(p.techniques)} for p in profiles]
with open(PROFILE_DIR / '_index.yaml', 'w') as f:
    yaml.dump(index, f, default_flow_style=False, sort_keys=False)

print(f"Saved {len(profiles)} profiles to {PROFILE_DIR}")
print(f"Files: {sorted(os.listdir(PROFILE_DIR))[:5]}...")

## 4. Simulation Experiments

Run each attacker profile against all defender strategies. The network is held constant (100 nodes, seed=42 for structure). Multiple simulation seeds provide statistical significance.

In [ ]:
# Experiment configuration
TOTAL_NODES = 100
FINISH_TIME = 15000
NUM_SEEDS = 5
SEEDS = [42, 123, 456, 789, 1024]

# Network configuration (constant across all experiments)
NETWORK_PARAMS = dict(
    total_nodes=TOTAL_NODES,
    total_endpoints=5,
    total_subnets=8,
    total_layers=4,
    total_database=2,
    terminate_compromise_ratio=0.8,
)

# Defender schemes to test
DEFENDER_SCHEMES = ['no_mtd', 'random', 'alternative', 'simultaneous']

# MTD AI configuration (if model available)
MODEL_PATH = "../src/mtdsim/ai/models/main_network_flagship.h5"
MTD_INTERVAL = 200
MTD_STRATEGIES = [CompleteTopologyShuffle, IPShuffle, OSDiversity, ServiceDiversity]
FEATURES = {
    "static": ["host_compromise_ratio", "attack_path_exposure", "overall_asr_avg", "roa", "risk"],
    "time": ["mtd_freq", "overall_mttc_avg", "time_since_last_mtd", "shortest_path_variability", "ip_variability", "attack_type"],
}

# Check if AI model is available
if AI_AVAILABLE and os.path.exists(MODEL_PATH):
    from tensorflow.keras.models import load_model
    trained_model = load_model(MODEL_PATH)
    DEFENDER_SCHEMES.append('mtd_ai')
    print(f"Loaded DDQN model from {MODEL_PATH}")
    print(f"Defender schemes: {DEFENDER_SCHEMES}")
else:
    trained_model = None
    print(f"DDQN model not found at {MODEL_PATH} — skipping MTD AI experiments.")
    print(f"Defender schemes: {DEFENDER_SCHEMES}")

print(f"\nExperimental matrix: {len(profiles)} campaigns x {len(DEFENDER_SCHEMES)} schemes x {NUM_SEEDS} seeds = {len(profiles) * len(DEFENDER_SCHEMES) * NUM_SEEDS} runs")

In [ ]:
def run_single_simulation(profile, scheme, seed):
    """Run one simulation with the given attacker profile and defender scheme.
    
    Returns a dict of metrics.
    """
    random.seed(seed)
    np.random.seed(seed)
    
    env = simpy.Environment()
    end_event = env.event()
    security_metrics = SecurityMetricStatistics()
    
    network = TimeNetwork(**NETWORK_PARAMS)
    adversary = Adversary(
        network=network,
        attack_threshold=constants.ATTACKER_THRESHOLD,
        profile=profile,
    )
    
    attack_op = AttackOperation(env=env, end_event=end_event, adversary=adversary, proceed_time=0)
    attack_op.proceed_attack()
    
    if scheme == 'mtd_ai' and trained_model is not None:
        mtd_op = MTDAIOperation(
            features=FEATURES,
            security_metrics_record=security_metrics,
            env=env, end_event=end_event,
            network=network, attack_operation=attack_op,
            scheme='mtd_ai', adversary=adversary,
            mtd_trigger_interval=MTD_INTERVAL,
            custom_strategies=MTD_STRATEGIES,
            main_network=trained_model,
            epsilon=0.01,
            attacker_sensitivity=1.0,
            static_degrade_factor=2000,
        )
        mtd_op.proceed_mtd()
    elif scheme != 'no_mtd':
        mtd_op = MTDOperation(
            security_metrics_record=security_metrics,
            env=env, end_event=end_event,
            network=network, attack_operation=attack_op,
            scheme=scheme, adversary=adversary,
        )
        mtd_op.proceed_mtd()
    # no_mtd: attack only
    
    env.run(until=FINISH_TIME)
    
    compromised = len(adversary.get_compromised_hosts())
    compromise_ratio = compromised / TOTAL_NODES
    
    # Extract attack statistics
    attack_record = adversary.get_statistics()
    
    # Time to first compromise
    compromise_records = attack_record[attack_record['compromise_host'].notna()] if len(attack_record) > 0 else pd.DataFrame()
    time_to_first = compromise_records['finish_time'].min() if len(compromise_records) > 0 else FINISH_TIME
    
    # Attack attempts
    total_attempts = adversary.get_curr_attempts()
    
    return {
        'campaign': profile.name,
        'campaign_id': profile.campaign_id,
        'scheme': scheme,
        'seed': seed,
        'compromised_hosts': compromised,
        'compromise_ratio': compromise_ratio,
        'time_to_first_compromise': time_to_first,
        'total_attempts': total_attempts,
        'sim_end_time': env.now,
    }

# Quick sanity check: run default profile vs no_mtd
test_result = run_single_simulation(AttackerProfile.default(), 'no_mtd', 42)
print(f"Sanity check (default profile, no MTD): {test_result['compromised_hosts']}/{TOTAL_NODES} compromised "
      f"({test_result['compromise_ratio']:.1%}) in {test_result['sim_end_time']:.0f}s")

In [ ]:
%%time

# Run the full experimental matrix
# Include a default (baseline) profile for comparison
all_profiles = [AttackerProfile.default()] + profiles
results = []

total_runs = len(all_profiles) * len(DEFENDER_SCHEMES) * NUM_SEEDS
pbar = tqdm(total=total_runs, desc="Running simulations")

for profile in all_profiles:
    for scheme in DEFENDER_SCHEMES:
        for seed in SEEDS:
            try:
                result = run_single_simulation(profile, scheme, seed)
                results.append(result)
            except Exception as e:
                print(f"ERROR: {profile.name} / {scheme} / seed={seed}: {e}")
                results.append({
                    'campaign': profile.name, 'campaign_id': profile.campaign_id,
                    'scheme': scheme, 'seed': seed,
                    'compromised_hosts': -1, 'compromise_ratio': -1,
                    'time_to_first_compromise': -1, 'total_attempts': -1,
                    'sim_end_time': -1,
                })
            pbar.update(1)

pbar.close()

results_df = pd.DataFrame(results)
results_df = results_df[results_df['compromised_hosts'] >= 0]  # drop errors

# Save raw results
RESULTS_DIR = Path("../experimental_data/campaign_experiments")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(RESULTS_DIR / "raw_results.csv", index=False)

print(f"\nCompleted {len(results_df)} successful runs.")
print(f"Results saved to {RESULTS_DIR / 'raw_results.csv'}")
display(results_df.head())

## 5. Results — Defender Scheme Comparison

For each defender scheme, compare attacker campaign performance.

In [ ]:
# Aggregate results: mean across seeds
agg_df = results_df.groupby(['campaign', 'scheme']).agg({
    'compromise_ratio': ['mean', 'std'],
    'time_to_first_compromise': ['mean', 'std'],
    'total_attempts': 'mean',
    'compromised_hosts': 'mean',
}).reset_index()

# Flatten multi-level columns
agg_df.columns = ['_'.join(col).strip('_') for col in agg_df.columns]

# Separate campaign profiles from baseline
campaign_agg = agg_df[agg_df['campaign'] != 'default']
baseline_agg = agg_df[agg_df['campaign'] == 'default']

print("Baseline (default attacker) results:")
display(baseline_agg[['scheme', 'compromise_ratio_mean', 'compromise_ratio_std', 'time_to_first_compromise_mean']].round(3))

In [ ]:
# Box plots: compromise ratio by scheme across campaigns
fig, ax = plt.subplots(figsize=(12, 6))
scheme_order = [s for s in DEFENDER_SCHEMES]
sns.boxplot(
    data=results_df[results_df['campaign'] != 'default'],
    x='scheme', y='compromise_ratio', order=scheme_order,
    palette='Set2', ax=ax,
)
# Overlay baseline as horizontal lines
for i, scheme in enumerate(scheme_order):
    baseline_val = baseline_agg[baseline_agg['scheme'] == scheme]['compromise_ratio_mean'].values
    if len(baseline_val) > 0:
        ax.hlines(baseline_val[0], i - 0.4, i + 0.4, colors='red', linestyles='dashed', linewidths=2)

ax.set_title('Compromise Ratio by Defender Scheme (all campaign profiles)\nRed dashed = default attacker baseline')
ax.set_ylabel('Compromise Ratio')
ax.set_xlabel('Defender Scheme')
plt.tight_layout()
plt.show()

In [ ]:
# Ranked bar chart: mean compromise ratio per campaign under no_mtd
no_mtd_agg = campaign_agg[campaign_agg['scheme'] == 'no_mtd'].sort_values('compromise_ratio_mean', ascending=True)

fig, ax = plt.subplots(figsize=(12, max(6, len(no_mtd_agg) * 0.3)))
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(no_mtd_agg)))
ax.barh(no_mtd_agg['campaign'], no_mtd_agg['compromise_ratio_mean'], 
        xerr=no_mtd_agg['compromise_ratio_std'], color=colors, edgecolor='gray')
ax.axvline(x=baseline_agg[baseline_agg['scheme'] == 'no_mtd']['compromise_ratio_mean'].values[0],
           color='red', linestyle='--', label='Default attacker')
ax.set_xlabel('Mean Compromise Ratio')
ax.set_title('Campaign Effectiveness (No MTD) — Higher = More Dangerous')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Time to first compromise comparison across schemes
fig = px.box(
    results_df[results_df['campaign'] != 'default'],
    x='scheme', y='time_to_first_compromise', color='scheme',
    title='Time to First Compromise by Defender Scheme (all campaigns)',
    labels={'time_to_first_compromise': 'Time to First Compromise (s)', 'scheme': 'Defender Scheme'},
    category_orders={'scheme': DEFENDER_SCHEMES},
)
fig.update_layout(height=500, showlegend=False)
fig.show()

## 6. Results — Campaign Deep-Dives

For the top campaigns by technique count, compare defender scheme effectiveness side-by-side.

In [ ]:
# Select top 6 campaigns by technique count + default baseline
top_campaign_names = [p.name for p in sorted(profiles, key=lambda p: len(p.techniques), reverse=True)[:6]]
focus_campaigns = ['default'] + top_campaign_names

focus_df = results_df[results_df['campaign'].isin(focus_campaigns)]
focus_agg = agg_df[agg_df['campaign'].isin(focus_campaigns)]

# Grouped bar chart: compromise ratio by campaign and scheme
fig = px.bar(
    focus_agg,
    x='campaign', y='compromise_ratio_mean',
    color='scheme', barmode='group',
    error_y='compromise_ratio_std',
    title='Compromise Ratio: Top Campaigns vs Defender Schemes',
    labels={'compromise_ratio_mean': 'Mean Compromise Ratio', 'campaign': 'Campaign'},
    category_orders={'scheme': DEFENDER_SCHEMES, 'campaign': focus_campaigns},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(height=500, xaxis_tickangle=-30)
fig.show()

In [ ]:
# Per-campaign subplots: scheme comparison
n_focus = len(top_campaign_names)
fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=True)

for idx, campaign_name in enumerate(top_campaign_names):
    ax = axes[idx // 3, idx % 3]
    camp_data = focus_agg[focus_agg['campaign'] == campaign_name]
    baseline_data = focus_agg[focus_agg['campaign'] == 'default']
    
    x = np.arange(len(DEFENDER_SCHEMES))
    width = 0.35
    
    camp_vals = [camp_data[camp_data['scheme'] == s]['compromise_ratio_mean'].values[0] 
                 if len(camp_data[camp_data['scheme'] == s]) > 0 else 0 for s in DEFENDER_SCHEMES]
    base_vals = [baseline_data[baseline_data['scheme'] == s]['compromise_ratio_mean'].values[0]
                 if len(baseline_data[baseline_data['scheme'] == s]) > 0 else 0 for s in DEFENDER_SCHEMES]
    
    ax.bar(x - width/2, camp_vals, width, label=campaign_name, color='coral')
    ax.bar(x + width/2, base_vals, width, label='Default', color='steelblue', alpha=0.7)
    
    ax.set_title(campaign_name, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([s.replace('_', '\n') for s in DEFENDER_SCHEMES], fontsize=8)
    ax.set_ylim(0, 1)
    if idx % 3 == 0:
        ax.set_ylabel('Compromise Ratio')
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle('Campaign vs Default Attacker: Compromise Ratio by Defender Scheme', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Cross-Analysis Heatmaps

Campaign x Defender heatmaps for key metrics.

In [ ]:
# Pivot table: campaign x scheme → mean compromise ratio
pivot_cr = campaign_agg.pivot_table(
    index='campaign', columns='scheme', values='compromise_ratio_mean'
)[DEFENDER_SCHEMES]

# Sort campaigns by mean across schemes (most dangerous first)
pivot_cr['mean_all'] = pivot_cr.mean(axis=1)
pivot_cr = pivot_cr.sort_values('mean_all', ascending=False).drop(columns='mean_all')

fig, ax = plt.subplots(figsize=(10, max(8, len(pivot_cr) * 0.35)))
sns.heatmap(
    pivot_cr, annot=True, fmt='.2f', cmap='RdYlGn_r',
    xticklabels=[s.replace('_', '\n') for s in DEFENDER_SCHEMES],
    ax=ax, vmin=0, vmax=1,
)
ax.set_title('Compromise Ratio: Campaign x Defender Scheme\n(0 = no compromise, 1 = fully compromised)')
ax.set_ylabel('Campaign (sorted by mean danger)')
plt.tight_layout()
plt.show()

In [ ]:
# Pivot table: time to first compromise
pivot_ttfc = campaign_agg.pivot_table(
    index='campaign', columns='scheme', values='time_to_first_compromise_mean'
)[DEFENDER_SCHEMES]

pivot_ttfc['mean_all'] = pivot_ttfc.mean(axis=1)
pivot_ttfc = pivot_ttfc.sort_values('mean_all', ascending=True).drop(columns='mean_all')

fig, ax = plt.subplots(figsize=(10, max(8, len(pivot_ttfc) * 0.35)))
sns.heatmap(
    pivot_ttfc, annot=True, fmt='.0f', cmap='RdYlGn',
    xticklabels=[s.replace('_', '\n') for s in DEFENDER_SCHEMES],
    ax=ax,
)
ax.set_title('Time to First Compromise (seconds): Campaign x Defender Scheme\n(lower = faster attacker)')
ax.set_ylabel('Campaign (sorted by speed)')
plt.tight_layout()
plt.show()

## 8. MTD AI Performance Analysis

How does the DDQN-based MTD agent (trained on default attacker) perform against diverse campaign profiles?
This tests whether the AI **generalises** to unseen attacker behaviours.

In [ ]:
if 'mtd_ai' in DEFENDER_SCHEMES:
    # Compare AI advantage over random scheme for each campaign
    ai_vs_random = campaign_agg.pivot_table(
        index='campaign', columns='scheme', values='compromise_ratio_mean'
    )
    
    if 'mtd_ai' in ai_vs_random.columns and 'random' in ai_vs_random.columns:
        ai_vs_random['ai_advantage'] = ai_vs_random['random'] - ai_vs_random['mtd_ai']
        ai_vs_random = ai_vs_random.sort_values('ai_advantage', ascending=True)
        
        fig, ax = plt.subplots(figsize=(12, max(6, len(ai_vs_random) * 0.3)))
        colors = ['green' if v > 0 else 'red' for v in ai_vs_random['ai_advantage']]
        ax.barh(ai_vs_random.index, ai_vs_random['ai_advantage'], color=colors, edgecolor='gray')
        ax.axvline(x=0, color='black', linewidth=0.5)
        ax.set_xlabel('AI Advantage (random CR - AI CR)')
        ax.set_title('DDQN MTD AI Advantage over Random Scheme per Campaign\n(Green = AI better, Red = Random better)')
        plt.tight_layout()
        plt.show()
        
        # Scatter: campaign "aggressiveness" vs AI advantage
        # Aggressiveness = mean capability across all phases
        aggr_scores = {}
        for p in profiles:
            aggr_scores[p.name] = np.mean(list(p.capability_profile.values()))
        
        scatter_data = ai_vs_random[['ai_advantage']].copy()
        scatter_data['aggressiveness'] = scatter_data.index.map(aggr_scores)
        scatter_data = scatter_data.dropna()
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.scatter(scatter_data['aggressiveness'], scatter_data['ai_advantage'], s=80, alpha=0.7)
        for name, row in scatter_data.iterrows():
            ax.annotate(name, (row['aggressiveness'], row['ai_advantage']), fontsize=8, alpha=0.8)
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Campaign Aggressiveness (mean capability)')
        ax.set_ylabel('AI Advantage over Random')
        ax.set_title('Does AI Generalise? Aggressiveness vs AI Advantage')
        plt.tight_layout()
        plt.show()
else:
    print("MTD AI experiments were skipped (model not available)."
          "\nRe-run after training the model with the retrain notebook.")

## 9. MTD Effectiveness Delta: How Much Does Each Scheme Reduce Campaign Damage?

Compare each MTD scheme's ability to reduce compromise relative to the no-MTD baseline, per campaign.

In [ ]:
# Compute MTD effectiveness delta: (no_mtd CR - scheme CR) / no_mtd CR
# Higher = more effective defense
mtd_schemes = [s for s in DEFENDER_SCHEMES if s != 'no_mtd']

pivot_all = campaign_agg.pivot_table(
    index='campaign', columns='scheme', values='compromise_ratio_mean'
)

delta_data = {}
for scheme in mtd_schemes:
    if scheme in pivot_all.columns and 'no_mtd' in pivot_all.columns:
        delta_data[scheme] = (pivot_all['no_mtd'] - pivot_all[scheme]) / pivot_all['no_mtd'].replace(0, np.nan)

delta_df = pd.DataFrame(delta_data)
delta_df = delta_df.sort_values(delta_df.columns[0], ascending=False) if len(delta_df.columns) > 0 else delta_df

fig, ax = plt.subplots(figsize=(10, max(8, len(delta_df) * 0.35)))
sns.heatmap(
    delta_df, annot=True, fmt='.1%', cmap='RdYlGn',
    xticklabels=[s.replace('_', '\n') for s in mtd_schemes],
    ax=ax, center=0,
)
ax.set_title('MTD Effectiveness: % Reduction in Compromise Ratio vs No-MTD Baseline\n(Green = effective defense, Red = MTD made things worse)')
ax.set_ylabel('Campaign')
plt.tight_layout()
plt.show()

# Summary statistics
print("\nMean MTD effectiveness across all campaigns:")
for scheme in mtd_schemes:
    if scheme in delta_df.columns:
        mean_eff = delta_df[scheme].mean()
        print(f"  {scheme}: {mean_eff:.1%} reduction in compromise")

## 10. Discussion & Observations

### Key Observations
*(To be filled after running the experiments)*

1. **Campaign Diversity**: The MITRE ATT&CK campaigns produce a meaningful range of attacker profiles, with some
   campaigns being significantly more capable in certain phases (e.g., exploitation-heavy vs. reconnaissance-heavy).

2. **MTD Effectiveness Varies by Attacker Profile**: Different MTD strategies may be more or less effective depending
   on the attacker's capability profile — a key finding for adaptive defense.

3. **AI Generalisation**: The DDQN model trained on default attacker parameters may/may not generalise well to
   campaign-derived profiles. This tests the robustness of RL-based MTD.

### Limitations

- **Tactic-to-phase mapping is a simplification**: The 14 ATT&CK tactics don't map 1:1 to MTDSim's kill chain.
  Some techniques span multiple tactics, and the mapping loses granularity.
- **Linear parameter derivation**: The formulas (duration = 1 - 0.5*C) are linear approximations. Real attacker
  capability may follow non-linear distributions.
- **Campaign technique counts as proxy for capability**: Having more techniques in a tactic doesn't necessarily
  mean higher capability — it could mean redundancy rather than skill.
- **Network randomness**: Despite fixed seeds for structure, the simulation still has stochastic elements
  (vulnerability generation, service assignment) that introduce variance.
- **AI model dependency**: If the DDQN model is not yet trained, AI comparisons are skipped.

### Future Work

- Retrain the DDQN on campaign-derived profiles and compare generalisation vs. specialisation.
- Incorporate technique-level weighting (e.g., using MITRE ATT&CK severity scores).
- Explore non-linear parameter derivation functions.
- Add time-series analysis of compromise progression during simulation.
- Cluster campaigns by capability profile and test representative archetypes.